# QML Observer -- Calibration & MVP Demo Benchmark

Milestone 7, Issue #52 ("Create benchmark notebook"). This notebook is a thin,
narrative wrapper around `benchmarks/run_benchmarks.py` -- every number here
comes from that module (Issues #53-#55), so the notebook and the CI-runnable
script never drift apart.

It covers:

1. The addendum §3 calibration sweep that picked `BarrenPlateauDetector`'s
   default `gradient_threshold` (Issue #54/#55b).
2. False-positive rate on healthy/convergence/noise fixtures (Issue #53).
3. Detection latency on the artificial-plateau fixture (Issue #54).
4. The combined convergence-vs-plateau comparison (Issue #55).
5. The blueprint Volume XX "critical MVP demo": a monitored run that stops
   early on a collapsed-gradient circuit while never stopping a healthy one.


In [1]:
import sys

sys.path.insert(0, "..")
sys.path.insert(0, "../tests")

from benchmarks.run_benchmarks import (
    _print_report,
    run_calibration_sweep,
    run_full_benchmark,
)

## 1. Calibration sweep (addendum §3)

The blueprint's `BarrenPlateauDetector` ships with placeholder thresholds.
Sweeping `gradient_threshold` against the fixture suite shows exactly where
the false-positive-free / plateau-detecting boundary sits.


In [2]:
sweep = run_calibration_sweep(
    candidate_gradient_thresholds=[1e-8, 1e-7, 1e-6, 2e-6, 5e-6, 1e-5],
    n_seeds=50,
)
for r in sweep:
    fp = r["false_positive"]
    lat = r["detection_latency"]
    worst_fp = max(v["false_positive_rate"] for v in fp.values())
    print(
        f"threshold={r['gradient_threshold']:.0e}  "
        f"worst_false_positive_rate={worst_fp:.1%}  "
        f"plateau_detection_rate={lat['detection_rate']:.1%}  "
        f"median_steps_to_detect={lat['median_steps_to_detection']}"
    )

threshold=1e-08  worst_false_positive_rate=0.0%  plateau_detection_rate=0.0%  median_steps_to_detect=None
threshold=1e-07  worst_false_positive_rate=0.0%  plateau_detection_rate=0.0%  median_steps_to_detect=None
threshold=1e-06  worst_false_positive_rate=0.0%  plateau_detection_rate=0.0%  median_steps_to_detect=None
threshold=2e-06  worst_false_positive_rate=0.0%  plateau_detection_rate=0.0%  median_steps_to_detect=None
threshold=5e-06  worst_false_positive_rate=0.0%  plateau_detection_rate=100.0%  median_steps_to_detect=14.0
threshold=1e-05  worst_false_positive_rate=0.0%  plateau_detection_rate=100.0%  median_steps_to_detect=14.0


`5e-6` is the first candidate with a 0% false-positive rate *and* a 100%
detection rate -- that is why it is the shipped default (see
`docs/research/validation.md` for the full writeup).

## 2 & 3. False positives and detection latency at the calibrated default


In [3]:
results = run_full_benchmark(n_seeds=50)
_print_report(results)

QML Observer — Calibration Benchmark
seeds=50  patience=15

False-positive rates (target: < 5%)
------------------------------------------------------------
  healthy_learning       0.0% (0/50)  [OK]
  convergence            0.0% (0/50)  [OK]
  noise_dominated        0.0% (0/50)  [OK]

Artificial-plateau detection latency (steps to first flag)
------------------------------------------------------------
  detection rate: 100.0% (50/50)
  median steps-to-detection: 14.0
  p95 steps-to-detection:    21


## 4. Convergence vs. plateau -- the comparison in one place

`results` above already bundles both halves (Issue #55): the
`false_positive` dict for healthy/convergence/noise fixtures, and the
`detection_latency` dict for the artificial-plateau fixture. Both were
generated by feeding the *same* seeded synthetic runs
(`tests/fixtures/synthetic_runs.py`) through the *same* detector/monitor
configuration, so they are directly comparable.


In [4]:
import json

print(json.dumps(results, indent=2, sort_keys=True))

{
  "config": {
    "n_seeds": 50,
    "patience": 15
  },
  "detection_latency": {
    "detection_rate": 1.0,
    "max_steps_to_detection": 29,
    "median_steps_to_detection": 14.0,
    "min_steps_to_detection": 14,
    "n_detected": 50,
    "n_seeds": 50,
    "p95_steps_to_detection": 21
  },
  "false_positive": {
    "convergence": {
      "false_positive_rate": 0.0,
      "meets_target_lt_5pct": true,
      "n_false_positive": 0,
      "n_seeds": 50
    },
    "healthy_learning": {
      "false_positive_rate": 0.0,
      "meets_target_lt_5pct": true,
      "n_false_positive": 0,
      "n_seeds": 50
    },
    "noise_dominated": {
      "false_positive_rate": 0.0,
      "meets_target_lt_5pct": true,
      "n_false_positive": 0,
      "n_seeds": 50
    }
  }
}


## 5. The critical MVP demo (blueprint Volume XX)

A live PennyLane run: a healthy circuit that must run to completion, and an
engineered collapsed-gradient circuit that must be stopped early with an
estimated-compute-saved figure. This is the same scenario shipped as
`examples/pennylane/barren_plateau_demo.py` (Milestone 6, Issue #46) --
skip this cell if PennyLane is not installed (`pip install qml-observer[pennylane]`).


In [5]:
try:
    import pennylane  # noqa: F401

    HAVE_PENNYLANE = True
except ImportError:
    HAVE_PENNYLANE = False

if HAVE_PENNYLANE:
    sys.path.insert(0, "../examples/pennylane")
    import barren_plateau_demo as demo

    demo.main()
else:
    print("PennyLane not installed -- skipping the live demo cell.")
    print("Install with: pip install qml-observer[pennylane]")


WITHOUT any issue -- healthy convergence (must NOT stop early)


Steps taken: 200 (planned budget: 5000)
Final diagnosis: converged (confidence=1.00)
Stopped early: False

WITH a collapsed-gradient run -- possible barren plateau (should stop early)
Steps taken: 15 (planned budget: 5000)
Final diagnosis: possible_barren_plateau (confidence=1.00)
Stopped early: True
Estimated compute saved: ~49.75s of wall-clock time (extrapolating 4985 unrun planned steps at ~9.98ms/step)

Summary
Healthy run stopped early:  False (expected: False)
Plateau run stopped early: True (expected: True)

Both expectations hold: this is the demonstration blueprint Volume XX asks for.
